# Предсказание на своей картинке

Модель выдаёт балл от 0 до 1 и карту, где она видит генерацию. Порог берётся из `runs/pilot/report.md`,
его пишет `scripts/run_test.py`: выше порога значит сгенерирована, при 1 % ложных срабатываний на реальных фото.

In [ ]:
import os, re, sys
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())

from pathlib import Path
import yaml, torch
import matplotlib.pyplot as plt
from src.model import Detector
from src.data import ImageDataset

cfg = yaml.safe_load(open('configs/train.yaml'))
out = Path(cfg['train']['out'])
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = Detector.load(out / 'best.pt', device)

# порог для каждого режима из отчёта теста; без отчёта берётся 0.5
report = out / 'report.md'
thresholds = {k: float(v) for k, v in re.findall(r'## protocol: (\w+)\s+threshold ([\d.]+)', report.read_text())} if report.exists() else {}
thresholds

Два режима, как в тесте; в обоих картинка сжимается в JPEG q90, как на обучении:
- `native` — исходное разрешение без уменьшения, как модель училась. Сторона больше 1024 пикселей обрезается по центру.
- `equalized` — картинка уменьшается и из центра вырезается 512×512. Проверка, что модель не опирается на размер.

In [ ]:
@torch.no_grad()
def predict(path, protocol='equalized'):
    item = ImageDataset([{'path': str(path), 'label': 0}], model.patch_size, protocol)[0]
    patch_logits, image_logit = model(item['image'][None].to(device))
    prob = torch.sigmoid(image_logit).item()
    return prob, prob > thresholds.get(protocol, 0.5), torch.sigmoid(patch_logits[0, 0]).cpu(), item['image']

In [ ]:
path = 'my_images/example.jpg'   # своя картинка
protocol = 'native'              # или 'equalized'

prob, fake, heat, image = predict(path, protocol)
print(f"{'сгенерирована' if fake else 'реальная'}: балл {prob:.3f}, порог {thresholds.get(protocol, 0.5):.3f}")
h, w = image.shape[1:]
fig, ax = plt.subplots(1, 2, figsize=(11, 5))
ax[0].imshow(image.permute(1, 2, 0))
ax[0].set_title(f"{'сгенерирована' if fake else 'реальная'}: {prob:.3f}, порог {thresholds.get(protocol, 0.5):.3f}")
ax[1].imshow(image.permute(1, 2, 0))
ax[1].imshow(heat, cmap='jet', alpha=0.45, vmin=0, vmax=1, extent=(0, w, h, 0))
ax[1].set_title('где модель видит генерацию')
for a in ax:
    a.axis('off')
plt.show()

In [ ]:
# все картинки из папки
folder = Path('my_images')       # папка, не файл
for p in sorted(folder.iterdir()):
    if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp'}:
        prob, fake, _, _ = predict(p, protocol)
        print(f"{prob:.3f}  {'FAKE' if fake else 'real'}  {p.name}")